In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path



In [2]:
# Define paths
DATA_PATH = Path("../../../../data/raw")

In [3]:
# Load data
df = pd.read_excel(DATA_PATH / "2010_Birth_Final.xlsx")

In [4]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

# CHANGED: Birth Weight column name (adjust based on your actual column name)
BIRTH_WEIGHT_COL = 'Birth_Weight(grams)'  # Change this to your actual column name
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH WEIGHT COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Weight Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_WEIGHT_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_WEIGHT_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
    print(f"\n💡 TIP: Please update BIRTH_WEIGHT_COL with the correct column name")
else:
    # Create missing birth weight indicator
    df['Missing_Birth_Weight'] = df[BIRTH_WEIGHT_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Weight'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth weight: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH WEIGHT ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Weight Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    # CHANGED: Now using Missing_Birth_Weight
    missing_weight = district_data['Missing_Birth_Weight'].sum()
    complete_weight = total_births - missing_weight
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_Weight': missing_weight,
        'Complete_Weight': complete_weight,
        'Missing_Rate': (missing_weight / total_births * 100) if total_births > 0 else 0,
        # Percentage of total missing records contributed by this district
        'Pct_of_Total_Missing': (missing_weight / total_missing * 100) if total_missing > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)
district_df = district_df.sort_values('Missing_Rate', ascending=False)

# NEW: Get top 15 districts by missing rate for detailed analysis
TOP_N_DISTRICTS = 15
top_districts_list = district_df.head(TOP_N_DISTRICTS)['District'].tolist()
print(f"\nTop {TOP_N_DISTRICTS} districts with highest missing rates:")
for idx, (_, row) in enumerate(district_df.head(TOP_N_DISTRICTS).iterrows(), 1):
    print(f"   {idx}. {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,} missing records)")

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT (TOP 15 DISTRICTS ONLY)
# ============================================

print(f"\n📊 STEP 4: Ethnicity Distribution by District (Top {TOP_N_DISTRICTS} Districts by Missing Rate)")
print("-" * 80)

# Create detailed ethnicity-district analysis - ONLY FOR TOP 15 DISTRICTS
district_ethnicity_analysis = []

for district in top_districts_list:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    district_missing_total = district_data['Missing_Birth_Weight'].sum()
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        # CHANGED: Now using Missing_Birth_Weight
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Weight'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            # Percentage of this district's missing records contributed by this ethnicity
            pct_of_district_missing_records = (ethnic_missing / district_missing_total * 100) if district_missing_total > 0 else 0
            
            # Percentage of total national missing records contributed by this ethnicity in this district
            pct_of_national_missing = (ethnic_missing / total_missing * 100) if total_missing > 0 else 0
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_Weight': ethnic_missing,  # CHANGED
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2),
                # NEW METRICS:
                'Pct_of_District_Missing_Records': round(pct_of_district_missing_records, 2),  # % of this district's missing
                'Pct_of_National_Missing': round(pct_of_national_missing, 2)  # % of total national missing
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title - CHANGED
    title = doc.add_heading('Missing Birth Weight Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 1: EXECUTIVE SUMMARY
    # ============================================
    
    doc.add_heading('1. Executive Summary', level=1)
    
    # CHANGED: Updated summary text for birth weight
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth weight data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Weight Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    • Focus Districts: Top {TOP_N_DISTRICTS} districts with highest missing rates
    
    Note: Birth weight is a critical indicator for newborn health, neonatal mortality, and 
    long-term developmental outcomes. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Public health intervention planning
    """
    
    doc.add_paragraph(summary_text)
    
    # ============================================
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    # ============================================
    
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 3: DISTRICT-WISE ANALYSIS (TOP 15 DISTRICTS)
    # ============================================
    
    doc.add_heading(f'3. District-Wise Missing Birth Weight Analysis (Top {TOP_N_DISTRICTS} Districts by Missing Rate)', level=1)
    
    # Add district summary table - CHANGED with new metrics
    doc.add_heading('3.1 District Summary Statistics', level=2)
    
    district_table = doc.add_table(rows=1, cols=6)
    district_table.style = 'Light Grid Accent 1'
    hdr_cells = district_table.rows[0].cells
    hdr_cells[0].text = 'District'
    hdr_cells[1].text = 'Total Births'
    hdr_cells[2].text = 'Missing Weight Records'
    hdr_cells[3].text = 'Missing Rate (%)'
    hdr_cells[4].text = '% of Total Missing Records'
    hdr_cells[5].text = 'Rank by Missing Rate'
    
    # Show only top 15 districts
    district_by_rate = district_df.head(TOP_N_DISTRICTS)
    
    for idx, (_, row) in enumerate(district_by_rate.iterrows(), 1):
        row_cells = district_table.add_row().cells
        row_cells[0].text = row['District']
        row_cells[1].text = f"{row['Total_Births']:,}"
        row_cells[2].text = f"{row['Missing_Weight']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[4].text = f"{row['Pct_of_Total_Missing']:.2f}%"
        row_cells[5].text = str(idx)
    
    # Add districts with highest missing counts among top 15
    doc.add_heading('3.2 Districts with Highest Missing Counts (Within Top 15 by Rate)', level=2)
    
    top_districts_by_count = district_by_rate.nlargest(10, 'Missing_Weight')
    count_table = doc.add_table(rows=1, cols=4)
    count_table.style = 'Light Grid Accent 1'
    hdr_cells = count_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Count'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Weight Count'
    hdr_cells[3].text = 'Missing Rate (%)'
    
    for idx, (_, row) in enumerate(top_districts_by_count.iterrows(), 1):
        row_cells = count_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Weight']:,}"
        row_cells[3].text = f"{row['Missing_Rate']:.2f}%"
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS (TOP 15 DISTRICTS)
    # ============================================
    
    doc.add_heading(f'4. Detailed District-Ethnicity Analysis (Top {TOP_N_DISTRICTS} Districts by Missing Rate)', level=1)
    doc.add_paragraph(f'The following tables show the ethnicity distribution and missing birth weight patterns for the {TOP_N_DISTRICTS} districts with the highest missing rates, following IUPAC nomenclature.')
    
    for district in top_districts_list[:TOP_N_DISTRICTS]:
        district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
        
        if len(district_ethnic) > 0:
            district_total = district_df[district_df['District'] == district]['Total_Births'].values[0]
            district_missing = district_df[district_df['District'] == district]['Missing_Weight'].values[0]
            district_pct_missing = district_df[district_df['District'] == district]['Pct_of_Total_Missing'].values[0]
            
            # Add district header
            doc.add_heading(f'{district}', level=2)
            doc.add_paragraph(f'Total Births: {district_total:,} | Missing Weight: {district_missing:,} ({district_missing/district_total*100:.2f}%) | {district_pct_missing:.1f}% of National Missing Records')
            
            # Create ethnicity table for district - CHANGED with new columns
            ethnic_table = doc.add_table(rows=1, cols=9)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'IUPAC Code'
            hdr_cells[1].text = 'Ethnicity'
            hdr_cells[2].text = 'Count'
            hdr_cells[3].text = '% of District'
            hdr_cells[4].text = 'Missing Weight'
            hdr_cells[5].text = 'Missing Rate (%)'
            hdr_cells[6].text = '% of District Missing Records'
            hdr_cells[7].text = '% of National Missing'
            hdr_cells[8].text = 'Rank in District'
            
            district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
            
            # Add rank by missing count within district
            district_ethnic_sorted = district_ethnic_sorted.copy()
            district_ethnic_sorted['Rank_In_District'] = district_ethnic_sorted['Missing_Weight'].rank(ascending=False, method='dense').astype(int)
            
            for _, row in district_ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['IUPAC_Code']
                row_cells[1].text = row['Ethnicity']
                row_cells[2].text = f"{row['Total_Mothers']:,}"
                row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                row_cells[4].text = f"{row['Missing_Weight']:,}"
                row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[6].text = f"{row['Pct_of_District_Missing_Records']:.2f}%"
                row_cells[7].text = f"{row['Pct_of_National_Missing']:.2f}%"
                row_cells[8].text = str(row['Rank_In_District'])
            
            # Add district summary
            doc.add_paragraph()
            most_prevalent = district_ethnic_sorted.iloc[0]
            highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing_Records'].idxmax()]
            largest_national_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_National_Missing'].idxmax()]
            
            summary_para = doc.add_paragraph()
            summary_para.add_run('District Summary:').bold = True
            doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
            doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing birth weight')
            doc.add_paragraph(f'• Largest contributor to district missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing_Records"]:.1f}% of district missing records')
            doc.add_paragraph(f'• Largest contributor to national missing data: {largest_national_contributor["Ethnicity"]} ({largest_national_contributor["IUPAC_Code"]}) - {largest_national_contributor["Pct_of_National_Missing"]:.1f}% of national missing records')
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 5: ETHNICITY-SPECIFIC ANALYSIS
    # ============================================
    
    doc.add_heading('5. Ethnicity-Specific Analysis', level=1)
    
    # Calculate overall ethnicity statistics - CHANGED with new metrics
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_Weight': 'sum',  # CHANGED
        'Pct_of_National_Missing': 'sum'  # NEW: Sum of national percentages
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_Weight'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    doc.add_heading('5.1 Overall Ethnicity Statistics', level=2)
    
    overall_table = doc.add_table(rows=1, cols=6)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Weight Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    hdr_cells[5].text = '% of National Missing'
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[5].text = f"{row['Pct_of_National_Missing']:.2f}%"
    
    # Add ethnicity-specific district analysis
    doc.add_heading('5.2 Ethnicity-Specific District Analysis (Top 15 Districts)', level=2)
    
    for ethnicity in list(ETHNICITIES.keys())[:6]:  # Show first 6 ethnicities
        ethnic_data = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
        
        if len(ethnic_data) > 0:
            ethnic_total = ethnic_data['Total_Mothers'].sum()
            ethnic_missing = ethnic_data['Missing_Weight'].sum()
            ethnic_rate = (ethnic_missing / ethnic_total * 100) if ethnic_total > 0 else 0
            ethnic_national_pct = ethnic_data['Pct_of_National_Missing'].sum()
            
            doc.add_heading(f'{ETHNICITIES[ethnicity]["full_name"]} ({ETHNICITIES[ethnicity]["code"]})', level=3)
            doc.add_paragraph(f'Overall Statistics: Total Births: {ethnic_total:,} | Missing Weight: {ethnic_missing:,} ({ethnic_rate:.2f}%) | {ethnic_national_pct:.1f}% of National Missing Records')
            
            # Create table for districts with highest missing rates AND highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Rates for this Ethnicity:', style='List Bullet')
            rate_table = doc.add_table(rows=1, cols=5)
            rate_table.style = 'Light Grid Accent 1'
            hdr_cells = rate_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_rate = ethnic_data.sort_values('Missing_Rate_in_Ethnic', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_rate.iterrows():
                row_cells = rate_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
            
            # NEW: Districts with highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Counts for this Ethnicity:', style='List Bullet')
            count_table = doc.add_table(rows=1, cols=5)
            count_table.style = 'Light Grid Accent 1'
            hdr_cells = count_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_count = ethnic_data.sort_values('Missing_Weight', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_count.iterrows():
                row_cells = count_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 6: CONCLUSIONS AND RECOMMENDATIONS
    # ============================================
    
    doc.add_heading('6. Conclusions and Recommendations', level=1)
    
    # Find key insights
    worst_district_by_rate = district_df.iloc[0]
    worst_district_by_count = district_df.nlargest(1, 'Missing_Weight').iloc[0]
    worst_ethnicity = ethnicity_overall.iloc[0] if len(ethnicity_overall) > 0 else None
    
    # Find which ethnicity contributes most to national missing
    top_national_contributor = ethnicity_overall.nlargest(1, 'Pct_of_National_Missing').iloc[0] if len(ethnicity_overall) > 0 else None
    
    # CHANGED: Updated conclusions for birth weight
    conclusions = f"""
    6.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth weight data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Geographic Disparities by Rate: {worst_district_by_rate['District']} shows the highest missing rate for birth weight at 
      {worst_district_by_rate['Missing_Rate']:.2f}%, suggesting potential data collection challenges in this region.
    
    • Geographic Disparities by Count: {worst_district_by_count['District']} has the highest absolute number of missing records
      ({worst_district_by_count['Missing_Weight']:,} records, representing {worst_district_by_count['Pct_of_Total_Missing']:.1f}% of all missing data).
    """
    
    if worst_ethnicity is not None:
        conclusions += f"""
    • Ethnic Disparities by Rate: {worst_ethnicity['Ethnicity']} ({worst_ethnicity['IUPAC_Code']}) has the highest 
      missing rate for birth weight at {worst_ethnicity['Missing_Rate']:.2f}%, indicating potential systematic bias 
      in data collection across ethnic groups.
        """
    
    if top_national_contributor is not None:
        conclusions += f"""
    • Ethnic Disparities by Contribution: {top_national_contributor['Ethnicity']} ({top_national_contributor['IUPAC_Code']}) 
      contributes the largest share ({top_national_contributor['Pct_of_National_Missing']:.1f}%) of all missing birth weight records nationally.
        """
    
    conclusions += """
    • Public Health Implications: Missing birth weight data affects the accuracy of low birth weight surveillance, 
      neonatal mortality assessments, and maternal and child health program evaluations. Birth weight is a critical 
      indicator for:
      - Low birth weight (<2500g) prevalence monitoring
      - Neonatal mortality risk assessment
      - Maternal nutrition intervention effectiveness
      - Newborn health outcomes tracking
    
    6.2 Recommendations
    
    1. Prioritize High-Volume Districts: Focus data quality improvement efforts first on districts with the highest
       absolute missing counts for birth weight.
    """
    
    conclusions += f"""
    2. Target High-Rate Districts: Implement specialized interventions in districts with the highest missing rates
       ({worst_district_by_rate['District']}: {worst_district_by_rate['Missing_Rate']:.1f}% missing).
    """
    
    if worst_ethnicity is not None:
        conclusions += f"""
    3. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with both high missing rates ({worst_ethnicity['Ethnicity']}: {worst_ethnicity['Missing_Rate']:.1f}%).
        """
    
    conclusions += """
    4. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    5. Regular Monitoring: Establish quarterly data quality monitoring systems to track improvements in 
       missing birth weight rates by district and ethnicity, focusing on both rates and absolute counts.
    
    6. Capacity Building: Provide training for healthcare workers on the importance of accurate birth weight 
       measurement and documentation, especially in high-volume and high-rate districts.
    
    7. Equipment Maintenance: Ensure regular calibration and maintenance of weighing scales in healthcare 
       facilities to prevent equipment-related missing data.
    
    8. Electronic Health Records: Implement or strengthen electronic health record systems with mandatory 
       birth weight fields to reduce missing data.
    """
    
    doc.add_paragraph(conclusions)
    
    # Add methodology section
    doc.add_heading('7. Methodology', level=1)
    
    methodology = f"""
    This analysis was conducted using vital statistics data from Sri Lanka. The methodology follows 
    IUPAC standards for ethnic classification and WHO guidelines for demographic data collection.
    
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Time Period: Full dataset analysis
    • Geographic Coverage: {len(districts)} districts
    • Focus Districts: Top {TOP_N_DISTRICTS} districts with highest missing rates
    
    Key Metrics Defined:
    
    1. Missing Rate: Percentage of records missing birth weight within a specific group
       Formula: (Missing Weight / Total Births) × 100
    
    2. Percentage of Total Missing Records: What proportion of ALL national missing records 
       comes from a specific district or ethnicity
       Formula: (Missing Weight in Group / Total National Missing) × 100
    
    3. Percentage of District Missing Records: What proportion of a district's missing records 
       comes from a specific ethnicity
       Formula: (Missing Weight for Ethnicity in District / Total District Missing) × 100
    
    4. Percentage of National Missing: What proportion of ALL national missing records comes 
       from a specific ethnicity-district combination
    
    Birth Weight Definition:
    Birth weight is the first weight of the newborn taken within the first hours of life, 
    measured in grams. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Very low birth weight tracking (<1500g)
    • Extremely low birth weight monitoring (<1000g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Demographic transition analysis
    • Newborn health outcomes research
    
    Birth Weight Categories (WHO):
    • Low Birth Weight: <2500g
    • Normal Birth Weight: 2500g - 3999g
    • High Birth Weight: ≥4000g
    
    Ethnic Classification:
    Ethnicities were classified according to IUPAC standards using the following codes:
    """
    
    doc.add_paragraph(methodology)
    
    # Add IUPAC code reference
    ref_table = doc.add_table(rows=1, cols=2)
    ref_table.style = 'Light Grid Accent 1'
    hdr_cells = ref_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnic Group'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = ref_table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
    
    # Save the document - CHANGED filename
    filename = f'Missing_Birth_Weight_Analysis_IUPAC_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # STEP 5: DISPLAY SUMMARY IN CONSOLE
    # ============================================
    
    print("\n" + "=" * 100)
    print(f"📊 FINAL SUMMARY: Missing Birth Weight Analysis by District and Ethnicity (Top {TOP_N_DISTRICTS} Districts)")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Focus Districts: {TOP_N_DISTRICTS} (highest missing rates)")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top {TOP_N_DISTRICTS} Districts with Highest Missing RATE:")
    for idx, (_, row) in enumerate(district_df.head(TOP_N_DISTRICTS).iterrows(), 1):
        print(f"   {idx}. {row['District']}: {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,}/{row['Total_Births']:,})")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing COUNT (absolute) among top {TOP_N_DISTRICTS}:")
    district_by_count = district_by_rate.nlargest(5, 'Missing_Weight')
    for _, row in district_by_count.iterrows():
        print(f"   • {row['District']}: {row['Missing_Weight']:,} missing records ({row['Missing_Rate']:.2f}% of district)")
    
    print(f"\n🏆 Top 5 Districts by % of Total National Missing among top {TOP_N_DISTRICTS}:")
    district_by_pct = district_by_rate.nlargest(5, 'Pct_of_Total_Missing')
    for _, row in district_by_pct.iterrows():
        print(f"   • {row['District']}: {row['Pct_of_Total_Missing']:.2f}% of national missing ({row['Missing_Weight']:,} records)")
    
    if len(ethnicity_overall) > 0:
        print(f"\n🏆 Top 5 Ethnicities with Highest Missing RATE (Overall):")
        for _, row in ethnicity_overall.head(5).iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,}/{row['Total_Mothers']:,})")
        
        print(f"\n🏆 Top 5 Ethnicities by % of Total National Missing:")
        ethnicity_by_pct = ethnicity_overall.nlargest(5, 'Pct_of_National_Missing')
        for _, row in ethnicity_by_pct.iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Pct_of_National_Missing']:.2f}% of national missing ({row['Missing_Weight']:,} records)")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 363,881
   • Missing birth weight: 45,303 (12.45%)
   • Complete birth weight: 318,578 (87.55%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Bharatha', 'Burgher', 'Indian Moor', 'Indian Tamil', 'Malay', 'Other Foreigners', 'Other Srilankans', 'Pakistan Moor', 'Sinhalese', 'Srilankan Chetty', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Top 15 districts with highest missing rates:
   1. Mullaitivu: 60.60% (769 missing records)
   2. Kandy: 39.80% (11,723 missing records)
   3. Gampaha: 29.54% (7,942 missing record

In [5]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

# CHANGED: Birth Weight column name (adjust based on your actual column name)
BIRTH_WEIGHT_COL = 'Birth_Weight(grams)'  # Change this to your actual column name
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH WEIGHT COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Weight Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_WEIGHT_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_WEIGHT_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
    print(f"\n💡 TIP: Please update BIRTH_WEIGHT_COL with the correct column name")
else:
    # Create missing birth weight indicator
    df['Missing_Birth_Weight'] = df[BIRTH_WEIGHT_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Weight'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth weight: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH WEIGHT ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Weight Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    missing_weight = district_data['Missing_Birth_Weight'].sum()
    complete_weight = total_births - missing_weight
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_Weight': missing_weight,
        'Complete_Weight': complete_weight,
        'Missing_Rate': (missing_weight / total_births * 100) if total_births > 0 else 0,
        # Percentage of total missing records contributed by this district
        'Pct_of_Total_Missing': (missing_weight / total_missing * 100) if total_missing > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)
district_df = district_df.sort_values('Missing_Rate', ascending=False)

# ============================================
# CRITICAL CHANGE: Select top districts by % of Total Missing Records
# NOT by Missing Rate
# ============================================

# Sort districts by % of total missing records (largest contributors first)
district_df_by_impact = district_df.sort_values('Pct_of_Total_Missing', ascending=False)

# Get top N districts by their contribution to total missing records
TOP_N_DISTRICTS = 15
top_districts_list = district_df_by_impact.head(TOP_N_DISTRICTS)['District'].tolist()

print(f"\nTop {TOP_N_DISTRICTS} districts by contribution to total missing records (most impactful):")
for idx, (_, row) in enumerate(district_df_by_impact.head(TOP_N_DISTRICTS).iterrows(), 1):
    print(f"   {idx}. {row['District']}: {row['Pct_of_Total_Missing']:.2f}% of total missing ({row['Missing_Weight']:,} missing records) - Missing Rate: {row['Missing_Rate']:.2f}%")

# Also show the districts with highest missing rates for context
print(f"\nFor context, districts with highest missing rates (but may have smaller impact):")
for idx, (_, row) in enumerate(district_df.head(5).iterrows(), 1):
    print(f"   {idx}. {row['District']}: {row['Missing_Rate']:.2f}% missing rate ({row['Missing_Weight']:,} records)")

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT (TOP IMPACT DISTRICTS ONLY)
# ============================================

print(f"\n📊 STEP 4: Ethnicity Distribution by District (Top {TOP_N_DISTRICTS} Districts by Impact)")
print("-" * 80)

# Create detailed ethnicity-district analysis - ONLY FOR TOP IMPACT DISTRICTS
district_ethnicity_analysis = []

for district in top_districts_list:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    district_missing_total = district_data['Missing_Birth_Weight'].sum()
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Weight'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            # Percentage of this district's missing records contributed by this ethnicity
            pct_of_district_missing_records = (ethnic_missing / district_missing_total * 100) if district_missing_total > 0 else 0
            
            # Percentage of total national missing records contributed by this ethnicity in this district
            pct_of_national_missing = (ethnic_missing / total_missing * 100) if total_missing > 0 else 0
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_Weight': ethnic_missing,
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2),
                'Pct_of_District_Missing_Records': round(pct_of_district_missing_records, 2),
                'Pct_of_National_Missing': round(pct_of_national_missing, 2)
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title
    title = doc.add_heading('Missing Birth Weight Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 1: EXECUTIVE SUMMARY - UPDATED
    # ============================================
    
    doc.add_heading('1. Executive Summary', level=1)
    
    # Calculate cumulative impact of top districts
    cumulative_missing = district_df_by_impact.head(TOP_N_DISTRICTS)['Missing_Weight'].sum()
    cumulative_pct = (cumulative_missing / total_missing * 100) if total_missing > 0 else 0
    
    # Updated summary text
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth weight data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Weight Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    • Focus Districts: Top {TOP_N_DISTRICTS} districts by contribution to total missing records
    
    CRITICAL INSIGHT: The top {TOP_N_DISTRICTS} districts contribute {cumulative_pct:.1f}% of all missing 
    birth weight records nationally ({cumulative_missing:,} out of {total_missing:,} total missing records).
    This concentration of missing data suggests that targeted interventions in these districts could 
    dramatically improve overall data completeness.
    
    Note: Birth weight is a critical indicator for newborn health, neonatal mortality, and 
    long-term developmental outcomes. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Public health intervention planning
    """
    
    doc.add_paragraph(summary_text)
    
    # ============================================
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    # ============================================
    
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 3: DISTRICT-WISE ANALYSIS - UPDATED
    # ============================================
    
    doc.add_heading(f'3. District-Wise Missing Birth Weight Analysis (Top {TOP_N_DISTRICTS} Districts by Impact)', level=1)
    doc.add_paragraph(f'The following analysis focuses on the {TOP_N_DISTRICTS} districts that contribute the most to the total missing birth weight records nationally. These districts represent the highest priority for data quality improvement interventions.')
    
    # Add district summary table - sorted by impact
    doc.add_heading('3.1 District Summary Statistics (Ranked by Impact)', level=2)
    
    district_table = doc.add_table(rows=1, cols=6)
    district_table.style = 'Light Grid Accent 1'
    hdr_cells = district_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Impact'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Weight Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    hdr_cells[5].text = '% of Total Missing Records'
    
    # Show only top impact districts
    district_by_impact = district_df_by_impact.head(TOP_N_DISTRICTS)
    
    for idx, (_, row) in enumerate(district_by_impact.iterrows(), 1):
        row_cells = district_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Total_Births']:,}"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[5].text = f"{row['Pct_of_Total_Missing']:.2f}%"
    
    # Add cumulative impact summary
    doc.add_heading('3.2 Cumulative Impact Analysis', level=2)
    
    cumulative_text = f"""
    The top 5 districts alone contribute {district_by_impact.head(5)['Pct_of_Total_Missing'].sum():.1f}% of all missing records,
    while the top {TOP_N_DISTRICTS} districts contribute {cumulative_pct:.1f}% of all missing records nationally.
    
    This concentration of missing data indicates that a targeted intervention strategy focusing on
    these high-impact districts could yield the greatest improvement in overall data completeness.
    """
    
    doc.add_paragraph(cumulative_text)
    
    # Add districts with highest missing rates for context
    doc.add_heading('3.3 Context: Districts with Highest Missing Rates', level=2)
    doc.add_paragraph('For reference, the following districts have the highest percentages of missing data, but may contribute less to the total missing count due to smaller population sizes:')
    
    rate_table = doc.add_table(rows=1, cols=4)
    rate_table.style = 'Light Grid Accent 1'
    hdr_cells = rate_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Rate'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Rate (%)'
    hdr_cells[3].text = 'Missing Count'
    
    for idx, (_, row) in enumerate(district_df.head(10).iterrows(), 1):
        row_cells = rate_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS
    # ============================================
    
    doc.add_heading(f'4. Detailed District-Ethnicity Analysis (Top {TOP_N_DISTRICTS} Districts by Impact)', level=1)
    doc.add_paragraph(f'The following tables show the ethnicity distribution and missing birth weight patterns for the {TOP_N_DISTRICTS} districts that contribute most to national missing records, following IUPAC nomenclature.')
    
    for district in top_districts_list[:TOP_N_DISTRICTS]:
        district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
        
        if len(district_ethnic) > 0:
            district_total = district_df_by_impact[district_df_by_impact['District'] == district]['Total_Births'].values[0]
            district_missing = district_df_by_impact[district_df_by_impact['District'] == district]['Missing_Weight'].values[0]
            district_pct_missing = district_df_by_impact[district_df_by_impact['District'] == district]['Pct_of_Total_Missing'].values[0]
            
            # Add district header with impact rank
            rank = district_df_by_impact[district_df_by_impact['District'] == district].index[0] + 1
            
            doc.add_heading(f'{district} (Impact Rank: #{rank})', level=2)
            doc.add_paragraph(f'Total Births: {district_total:,} | Missing Weight: {district_missing:,} ({district_missing/district_total*100:.2f}%) | {district_pct_missing:.1f}% of National Missing Records')
            
            # Create ethnicity table for district
            ethnic_table = doc.add_table(rows=1, cols=9)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'IUPAC Code'
            hdr_cells[1].text = 'Ethnicity'
            hdr_cells[2].text = 'Count'
            hdr_cells[3].text = '% of District'
            hdr_cells[4].text = 'Missing Weight'
            hdr_cells[5].text = 'Missing Rate (%)'
            hdr_cells[6].text = '% of District Missing Records'
            hdr_cells[7].text = '% of National Missing'
            hdr_cells[8].text = 'Rank in District'
            
            district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
            
            # Add rank by missing count within district
            district_ethnic_sorted = district_ethnic_sorted.copy()
            district_ethnic_sorted['Rank_In_District'] = district_ethnic_sorted['Missing_Weight'].rank(ascending=False, method='dense').astype(int)
            
            for _, row in district_ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['IUPAC_Code']
                row_cells[1].text = row['Ethnicity']
                row_cells[2].text = f"{row['Total_Mothers']:,}"
                row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                row_cells[4].text = f"{row['Missing_Weight']:,}"
                row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[6].text = f"{row['Pct_of_District_Missing_Records']:.2f}%"
                row_cells[7].text = f"{row['Pct_of_National_Missing']:.2f}%"
                row_cells[8].text = str(row['Rank_In_District'])
            
            # Add district summary
            doc.add_paragraph()
            most_prevalent = district_ethnic_sorted.iloc[0]
            highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing_Records'].idxmax()]
            largest_national_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_National_Missing'].idxmax()]
            
            summary_para = doc.add_paragraph()
            summary_para.add_run('District Summary:').bold = True
            doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
            doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing birth weight')
            doc.add_paragraph(f'• Largest contributor to district missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing_Records"]:.1f}% of district missing records')
            doc.add_paragraph(f'• Largest contributor to national missing data: {largest_national_contributor["Ethnicity"]} ({largest_national_contributor["IUPAC_Code"]}) - {largest_national_contributor["Pct_of_National_Missing"]:.1f}% of national missing records')
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 5: ETHNICITY-SPECIFIC ANALYSIS
    # ============================================
    
    doc.add_heading('5. Ethnicity-Specific Analysis', level=1)
    
    # Calculate overall ethnicity statistics
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_Weight': 'sum',
        'Pct_of_National_Missing': 'sum'
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_Weight'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    doc.add_heading('5.1 Overall Ethnicity Statistics', level=2)
    
    overall_table = doc.add_table(rows=1, cols=6)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Weight Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    hdr_cells[5].text = '% of National Missing'
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[5].text = f"{row['Pct_of_National_Missing']:.2f}%"
    
    # Add ethnicity-specific district analysis
    doc.add_heading('5.2 Ethnicity-Specific District Analysis (Top Impact Districts)', level=2)
    
    for ethnicity in list(ETHNICITIES.keys())[:6]:
        ethnic_data = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
        
        if len(ethnic_data) > 0:
            ethnic_total = ethnic_data['Total_Mothers'].sum()
            ethnic_missing = ethnic_data['Missing_Weight'].sum()
            ethnic_rate = (ethnic_missing / ethnic_total * 100) if ethnic_total > 0 else 0
            ethnic_national_pct = ethnic_data['Pct_of_National_Missing'].sum()
            
            doc.add_heading(f'{ETHNICITIES[ethnicity]["full_name"]} ({ETHNICITIES[ethnicity]["code"]})', level=3)
            doc.add_paragraph(f'Overall Statistics: Total Births: {ethnic_total:,} | Missing Weight: {ethnic_missing:,} ({ethnic_rate:.2f}%) | {ethnic_national_pct:.1f}% of National Missing Records')
            
            # Create table for districts with highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Counts for this Ethnicity:', style='List Bullet')
            count_table = doc.add_table(rows=1, cols=5)
            count_table.style = 'Light Grid Accent 1'
            hdr_cells = count_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_count = ethnic_data.sort_values('Missing_Weight', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_count.iterrows():
                row_cells = count_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
            
            # Table for districts with highest missing rates for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Rates for this Ethnicity:', style='List Bullet')
            rate_table = doc.add_table(rows=1, cols=5)
            rate_table.style = 'Light Grid Accent 1'
            hdr_cells = rate_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_rate = ethnic_data.sort_values('Missing_Rate_in_Ethnic', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_rate.iterrows():
                row_cells = rate_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 6: CONCLUSIONS AND RECOMMENDATIONS - UPDATED
    # ============================================
    
    doc.add_heading('6. Conclusions and Recommendations', level=1)
    
    # Find key insights
    top_impact_district = district_df_by_impact.iloc[0]
    top_5_impact = district_df_by_impact.head(5)
    top_5_cumulative_pct = top_5_impact['Pct_of_Total_Missing'].sum()
    
    worst_ethnicity_by_rate = ethnicity_overall.iloc[0] if len(ethnicity_overall) > 0 else None
    top_ethnic_contributor = ethnicity_overall.nlargest(1, 'Pct_of_National_Missing').iloc[0] if len(ethnicity_overall) > 0 else None
    
    # Updated conclusions
    conclusions = f"""
    6.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth weight data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Concentration of Missing Data: The top {TOP_N_DISTRICTS} districts contribute {cumulative_pct:.1f}% of all missing 
      birth weight records nationally. The top 5 districts alone contribute {top_5_cumulative_pct:.1f}% of all missing records.
      This high concentration suggests that targeted interventions could dramatically improve overall data completeness.
    
    • Highest Impact District: {top_impact_district['District']} contributes the largest share of missing records 
      ({top_impact_district['Pct_of_Total_Missing']:.1f}% of total missing, {top_impact_district['Missing_Weight']:,} records).
      This district should be the highest priority for intervention.
    """
    
    if worst_ethnicity_by_rate is not None:
        conclusions += f"""
    • Ethnic Disparities by Rate: {worst_ethnicity_by_rate['Ethnicity']} ({worst_ethnicity_by_rate['IUPAC_Code']}) has the highest 
      missing rate for birth weight at {worst_ethnicity_by_rate['Missing_Rate']:.2f}%, indicating potential systematic bias 
      in data collection across ethnic groups.
        """
    
    if top_ethnic_contributor is not None:
        conclusions += f"""
    • Ethnic Disparities by Contribution: {top_ethnic_contributor['Ethnicity']} ({top_ethnic_contributor['IUPAC_Code']}) 
      contributes the largest share ({top_ethnic_contributor['Pct_of_National_Missing']:.1f}%) of all missing birth weight records nationally.
        """
    
    conclusions += """
    • Public Health Implications: Missing birth weight data affects the accuracy of low birth weight surveillance, 
      neonatal mortality assessments, and maternal and child health program evaluations. Birth weight is a critical 
      indicator for:
      - Low birth weight (<2500g) prevalence monitoring
      - Neonatal mortality risk assessment
      - Maternal nutrition intervention effectiveness
      - Newborn health outcomes tracking
    
    6.2 Recommendations (Prioritized by Impact)
    
    1. Focus on High-Impact Districts: Prioritize data quality improvement efforts on the top 5 districts that
       contribute the largest share of missing records. These districts alone account for {top_5_cumulative_pct:.1f}% 
       of all missing birth weight data nationally.
    """
    
    conclusions += f"""
    2. District-Specific Interventions: Implement specialized data collection protocols in the highest impact districts:
       • {top_impact_district['District']}: Highest priority ({top_impact_district['Pct_of_Total_Missing']:.1f}% of total missing)
    """
    
    # Add top 3 impact districts
    for idx, (_, row) in enumerate(top_5_impact.iterrows(), 1):
        conclusions += f"       • {row['District']}: {row['Pct_of_Total_Missing']:.1f}% of total missing\n"
    
    if worst_ethnicity_by_rate is not None:
        conclusions += f"""
    3. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with both high missing rates and high contribution to national missing data, focusing on:
       • {worst_ethnicity_by_rate['Ethnicity']}: {worst_ethnicity_by_rate['Missing_Rate']:.1f}% missing rate
       • {top_ethnic_contributor['Ethnicity']}: {top_ethnic_contributor['Pct_of_National_Missing']:.1f}% of national missing
        """
    
    conclusions += """
    4. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    5. Regular Monitoring: Establish quarterly data quality monitoring systems with a focus on the high-impact
       districts identified in this report.
    
    6. Capacity Building: Provide training for healthcare workers on the importance of accurate birth weight 
       measurement and documentation, with intensified efforts in high-impact districts.
    
    7. Equipment Maintenance: Ensure regular calibration and maintenance of weighing scales in healthcare 
       facilities, prioritizing high-volume facilities in high-impact districts.
    
    8. Electronic Health Records: Implement or strengthen electronic health record systems with mandatory 
       birth weight fields, with phased implementation starting in high-impact districts.
    """
    
    doc.add_paragraph(conclusions)
    
    # Add methodology section
    doc.add_heading('7. Methodology', level=1)
    
    methodology = f"""
    This analysis was conducted using vital statistics data from Sri Lanka. The methodology follows 
    IUPAC standards for ethnic classification and WHO guidelines for demographic data collection.
    
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Time Period: Full dataset analysis
    • Geographic Coverage: {len(districts)} districts
    • Focus Districts: Top {TOP_N_DISTRICTS} districts by contribution to total missing records
    
    Key Metrics Defined:
    
    1. Missing Rate: Percentage of records missing birth weight within a specific group
       Formula: (Missing Weight / Total Births) × 100
    
    2. Percentage of Total Missing Records: What proportion of ALL national missing records 
       comes from a specific district or ethnicity
       Formula: (Missing Weight in Group / Total National Missing) × 100
    
    IMPORTANT: Districts were selected for detailed analysis based on their contribution to 
    total missing records (Percentage of Total Missing Records), NOT by Missing Rate. This 
    ensures that the analysis focuses on districts that have the greatest potential impact 
    on overall data completeness.
    
    3. Percentage of District Missing Records: What proportion of a district's missing records 
       comes from a specific ethnicity
       Formula: (Missing Weight for Ethnicity in District / Total District Missing) × 100
    
    4. Percentage of National Missing: What proportion of ALL national missing records comes 
       from a specific ethnicity-district combination
    
    Birth Weight Definition:
    Birth weight is the first weight of the newborn taken within the first hours of life, 
    measured in grams. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Very low birth weight tracking (<1500g)
    • Extremely low birth weight monitoring (<1000g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Demographic transition analysis
    • Newborn health outcomes research
    
    Birth Weight Categories (WHO):
    • Low Birth Weight: <2500g
    • Normal Birth Weight: 2500g - 3999g
    • High Birth Weight: ≥4000g
    
    Ethnic Classification:
    Ethnicities were classified according to IUPAC standards using the following codes:
    """
    
    doc.add_paragraph(methodology)
    
    # Add IUPAC code reference
    ref_table = doc.add_table(rows=1, cols=2)
    ref_table.style = 'Light Grid Accent 1'
    hdr_cells = ref_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnic Group'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = ref_table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
    
    # Save the document
    filename = f'Missing_Birth_Weight_Analysis_IUPAC_Impact_Based_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # STEP 5: DISPLAY SUMMARY IN CONSOLE - UPDATED
    # ============================================
    
    print("\n" + "=" * 100)
    print(f"📊 FINAL SUMMARY: Missing Birth Weight Analysis by District and Ethnicity")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Focus Districts: {TOP_N_DISTRICTS} (by contribution to total missing)")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top {TOP_N_DISTRICTS} Districts by IMPACT (% of Total Missing Records):")
    for idx, (_, row) in enumerate(district_df_by_impact.head(TOP_N_DISTRICTS).iterrows(), 1):
        print(f"   {idx}. {row['District']}: {row['Pct_of_Total_Missing']:.2f}% of total missing ({row['Missing_Weight']:,} records) - Missing Rate: {row['Missing_Rate']:.2f}%")
    
    print(f"\n📊 Cumulative Impact:")
    print(f"   • Top 5 districts account for {district_df_by_impact.head(5)['Pct_of_Total_Missing'].sum():.1f}% of all missing records")
    print(f"   • Top 10 districts account for {district_df_by_impact.head(10)['Pct_of_Total_Missing'].sum():.1f}% of all missing records")
    print(f"   • Top {TOP_N_DISTRICTS} districts account for {cumulative_pct:.1f}% of all missing records")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing RATE (for context):")
    for _, row in district_df.head(5).iterrows():
        print(f"   • {row['District']}: {row['Missing_Rate']:.2f}% missing rate ({row['Missing_Weight']:,} records)")
    
    if len(ethnicity_overall) > 0:
        print(f"\n🏆 Top 5 Ethnicities with Highest Missing RATE (Overall):")
        for _, row in ethnicity_overall.head(5).iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,}/{row['Total_Mothers']:,})")
        
        print(f"\n🏆 Top 5 Ethnicities by % of Total National Missing:")
        ethnicity_by_pct = ethnicity_overall.nlargest(5, 'Pct_of_National_Missing')
        for _, row in ethnicity_by_pct.iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Pct_of_National_Missing']:.2f}% of national missing ({row['Missing_Weight']:,} records)")
    
    print(f"\n💡 KEY INSIGHT: Focus interventions on the top {TOP_N_DISTRICTS} districts to address {cumulative_pct:.1f}% of all missing birth weight records.")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 363,881
   • Missing birth weight: 45,303 (12.45%)
   • Complete birth weight: 318,578 (87.55%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Bharatha', 'Burgher', 'Indian Moor', 'Indian Tamil', 'Malay', 'Other Foreigners', 'Other Srilankans', 'Pakistan Moor', 'Sinhalese', 'Srilankan Chetty', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Top 15 districts by contribution to total missing records (most impactful):
   1. Kandy: 25.88% of total missing (11,723 missing records) - Missing Rate: 39.80%
   2. Gampaha: 17.5

In [6]:
# ============================================
# INSTALL REQUIRED PACKAGES (if not already installed)
# ============================================
try:
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement
except ImportError:
    print("Installing python-docx...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'python-docx'])
    from docx import Document
    from docx.shared import Inches, Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from docx.enum.table import WD_TABLE_ALIGNMENT
    from docx.oxml.ns import qn
    from docx.oxml import OxmlElement

import pandas as pd
import numpy as np
from datetime import datetime

# ============================================
# SET CORRECT COLUMN NAMES
# ============================================

# CHANGED: Birth Weight column name (adjust based on your actual column name)
BIRTH_WEIGHT_COL = 'Birth_Weight(grams)'  # Change this to your actual column name
MOTHER_RACE_COL = 'Race_of_Mother'
FATHER_RACE_COL = 'Race_of_Father'
DISTRICT_COL = 'Registered_District'

print("=" * 80)
print("🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY")
print("=" * 80)

# ============================================
# STEP 1: IDENTIFY MISSING BIRTH WEIGHT COUNTS
# ============================================

print("\n📊 STEP 1: Missing Birth Weight Analysis")
print("-" * 80)

# Check if column exists
if BIRTH_WEIGHT_COL not in df.columns:
    print(f"❌ ERROR: Column '{BIRTH_WEIGHT_COL}' not found!")
    print(f"Available columns: {list(df.columns)}")
    print(f"\n💡 TIP: Please update BIRTH_WEIGHT_COL with the correct column name")
else:
    # Create missing birth weight indicator
    df['Missing_Birth_Weight'] = df[BIRTH_WEIGHT_COL].isna()
    
    # Overall missing statistics
    total_records = len(df)
    total_missing = df['Missing_Birth_Weight'].sum()
    total_complete = total_records - total_missing
    
    print(f"\nOverall Statistics:")
    print(f"   • Total records: {total_records:,}")
    print(f"   • Missing birth weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    print(f"   • Complete birth weight: {total_complete:,} ({total_complete/total_records*100:.2f}%)")

# ============================================
# STEP 2: ETHNICITY STANDARDIZATION (IUPAC STANDARDS)
# ============================================

print("\n📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)")
print("-" * 80)

# Define ethnicities with IUPAC codes following international standards
ETHNICITIES = {
    'Sinhalese': {'code': 'SIN', 'full_name': 'Sinhalese', 'iupac_name': 'Sinhala'},
    'Srilankan Tamil': {'code': 'TAM_SL', 'full_name': 'Sri Lankan Tamil', 'iupac_name': 'Tamil (Sri Lanka)'},
    'Indian Tamil': {'code': 'TAM_IN', 'full_name': 'Indian Tamil', 'iupac_name': 'Tamil (India)'},
    'Srilankan Moor': {'code': 'MOOR_SL', 'full_name': 'Sri Lankan Moor', 'iupac_name': 'Moor (Sri Lanka)'},
    'Burgher': {'code': 'BUR', 'full_name': 'Burgher', 'iupac_name': 'Burgher'},
    'Malay': {'code': 'MAL', 'full_name': 'Malay', 'iupac_name': 'Malay'},
    'Srilankan Chetty': {'code': 'CHT_SL', 'full_name': 'Sri Lankan Chetty', 'iupac_name': 'Chetty'},
    'Bharatha': {'code': 'BHA', 'full_name': 'Bharatha', 'iupac_name': 'Bharatha'},
    'Indian Moor': {'code': 'MOOR_IN', 'full_name': 'Indian Moor', 'iupac_name': 'Moor (India)'},
    'Pakistan Moor': {'code': 'MOOR_PK', 'full_name': 'Pakistan Moor', 'iupac_name': 'Moor (Pakistan)'},
    'Other Foreigners': {'code': 'OTH_FGN', 'full_name': 'Other Foreigners', 'iupac_name': 'Other Nationalities'},
    'Other Srilankans': {'code': 'OTH_SL', 'full_name': 'Other Sri Lankans', 'iupac_name': 'Other Ethnic Groups'}
}

# Standardize ethnicity function
def standardize_ethnicity(race):
    """Standardize ethnicity names following IUPAC nomenclature"""
    if pd.isna(race):
        return None
    
    # Convert to string and strip
    race_str = str(race).strip()
    
    # Mapping for numeric codes (1-13) as per Sri Lanka vital statistics
    code_map = {
        '1': 'Sinhalese',
        '2': 'Srilankan Tamil',
        '3': 'Indian Tamil',
        '4': 'Srilankan Moor',
        '5': 'Burgher',
        '6': 'Malay',
        '7': 'Srilankan Chetty',
        '8': 'Bharatha',
        '9': 'Indian Moor',
        '10': 'Pakistan Moor',
        '11': 'Other Foreigners',
        '12': 'Other Srilankans'
    }
    
    # Mapping for text values
    text_map = {
        'Sinhalese': 'Sinhalese',
        'Srilankan Tamil': 'Srilankan Tamil',
        'Sri Lankan Tamil': 'Srilankan Tamil',
        'Indian Tamil': 'Indian Tamil',
        'Srilankan Moor': 'Srilankan Moor',
        'Sri Lankan Moor': 'Srilankan Moor',
        'Moor': 'Srilankan Moor',
        'Burgher': 'Burgher',
        'Malay': 'Malay',
        'Srilankan Chetty': 'Srilankan Chetty',
        'Bharatha': 'Bharatha',
        'Indian Moor': 'Indian Moor',
        'Pakistan Moor': 'Pakistan Moor',
        'Other Foreigners': 'Other Foreigners',
        'Other Srilankans': 'Other Srilankans'
    }
    
    # Check if it's a numeric code
    if race_str in code_map:
        return code_map[race_str]
    
    # Check if it's a text value
    return text_map.get(race_str, None)

# Create standardized ethnicity columns
df['Mother_Ethnicity_Std'] = df[MOTHER_RACE_COL].apply(standardize_ethnicity)
df['Father_Ethnicity_Std'] = df[FATHER_RACE_COL].apply(standardize_ethnicity)

# Show unique ethnicities found
unique_ethnicities = df['Mother_Ethnicity_Std'].dropna().unique()
print(f"\nUnique ethnicities found in dataset: {sorted(unique_ethnicities)}")

# ============================================
# STEP 3: DISTRICT-LEVEL MISSING BIRTH WEIGHT ANALYSIS
# ============================================

print("\n📊 STEP 3: District-Level Missing Birth Weight Analysis")
print("-" * 80)

# Get unique districts
districts = df[DISTRICT_COL].dropna().unique()
district_summary = []

for district in districts:
    district_data = df[df[DISTRICT_COL] == district]
    
    total_births = len(district_data)
    missing_weight = district_data['Missing_Birth_Weight'].sum()
    complete_weight = total_births - missing_weight
    
    district_summary.append({
        'District': district,
        'Total_Births': total_births,
        'Missing_Weight': missing_weight,
        'Complete_Weight': complete_weight,
        'Missing_Rate': (missing_weight / total_births * 100) if total_births > 0 else 0,
        # Percentage of total missing records contributed by this district
        'Pct_of_Total_Missing': (missing_weight / total_missing * 100) if total_missing > 0 else 0
    })

# Create district summary DataFrame
district_df = pd.DataFrame(district_summary)

# Create two sorted versions:
# 1. Sorted by Missing Rate (for reference)
district_df_by_rate = district_df.sort_values('Missing_Rate', ascending=False)

# 2. Sorted by Impact (% of Total Missing Records) for selection of focus districts
district_df_by_impact = district_df.sort_values('Pct_of_Total_Missing', ascending=False)

# Get top N districts by their contribution to total missing records for detailed analysis
TOP_N_DISTRICTS = 15
top_districts_list = district_df_by_impact.head(TOP_N_DISTRICTS)['District'].tolist()

print(f"\nAll Districts Summary (Total: {len(districts)} districts):")
print(f"   • Total records across all districts: {total_records:,}")
print(f"   • Total missing records: {total_missing:,}")
print(f"\nTop {TOP_N_DISTRICTS} districts by IMPACT (% of Total Missing Records):")
for idx, (_, row) in enumerate(district_df_by_impact.head(TOP_N_DISTRICTS).iterrows(), 1):
    print(f"   {idx}. {row['District']}: {row['Pct_of_Total_Missing']:.2f}% of total missing ({row['Missing_Weight']:,} missing records) - Missing Rate: {row['Missing_Rate']:.2f}%")

print(f"\nFor context, districts with highest missing rates (but may have smaller impact):")
for idx, (_, row) in enumerate(district_df_by_rate.head(5).iterrows(), 1):
    print(f"   {idx}. {row['District']}: {row['Missing_Rate']:.2f}% missing rate ({row['Missing_Weight']:,} records)")

# ============================================
# STEP 4: ETHNICITY DISTRIBUTION BY DISTRICT (TOP IMPACT DISTRICTS ONLY)
# ============================================

print(f"\n📊 STEP 4: Ethnicity Distribution by District (Top {TOP_N_DISTRICTS} Districts by Impact)")
print("-" * 80)

# Create detailed ethnicity-district analysis - ONLY FOR TOP IMPACT DISTRICTS
district_ethnicity_analysis = []

for district in top_districts_list:
    district_data = df[df[DISTRICT_COL] == district]
    district_total = len(district_data)
    district_missing_total = district_data['Missing_Birth_Weight'].sum()
    
    if district_total == 0:
        continue
    
    for ethnicity, config in ETHNICITIES.items():
        ethnic_mothers = district_data[district_data['Mother_Ethnicity_Std'] == ethnicity].shape[0]
        ethnic_missing = district_data[
            (district_data['Mother_Ethnicity_Std'] == ethnicity) & 
            (district_data['Missing_Birth_Weight'] == True)
        ].shape[0]
        
        if ethnic_mothers > 0:
            pct_of_district = (ethnic_mothers / district_total * 100)
            pct_of_district_missing = (ethnic_missing / district_total * 100)
            missing_rate_in_ethnic = (ethnic_missing / ethnic_mothers * 100)
            
            # Percentage of this district's missing records contributed by this ethnicity
            pct_of_district_missing_records = (ethnic_missing / district_missing_total * 100) if district_missing_total > 0 else 0
            
            # Percentage of total national missing records contributed by this ethnicity in this district
            pct_of_national_missing = (ethnic_missing / total_missing * 100) if total_missing > 0 else 0
            
            district_ethnicity_analysis.append({
                'District': district,
                'Ethnicity': ethnicity,
                'IUPAC_Code': config['code'],
                'IUPAC_Name': config['iupac_name'],
                'Total_Mothers': ethnic_mothers,
                'Pct_of_District_Total': round(pct_of_district, 2),
                'Missing_Weight': ethnic_missing,
                'Missing_Rate_in_Ethnic': round(missing_rate_in_ethnic, 2),
                'Pct_of_District_Missing': round(pct_of_district_missing, 2),
                'Pct_of_District_Missing_Records': round(pct_of_district_missing_records, 2),
                'Pct_of_National_Missing': round(pct_of_national_missing, 2)
            })

# Create DataFrame
if district_ethnicity_analysis:
    ethnicity_district_df = pd.DataFrame(district_ethnicity_analysis)
    
    # ============================================
    # CREATE WORD DOCUMENT WITH IUPAC STANDARDS
    # ============================================
    
    print("\n📄 Creating Word Document with IUPAC Standards...")
    
    # Create new document
    doc = Document()
    
    # Set document margins
    sections = doc.sections
    for section in sections:
        section.top_margin = Inches(1)
        section.bottom_margin = Inches(1)
        section.left_margin = Inches(1)
        section.right_margin = Inches(1)
    
    # Add title
    title = doc.add_heading('Missing Birth Weight Analysis Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add subtitle with IUPAC standards
    subtitle = doc.add_heading('Sri Lanka Vital Statistics - IUPAC Compliant Nomenclature', 2)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Add date
    date_para = doc.add_paragraph(f'Report Generated: {datetime.now().strftime("%B %d, %Y")}')
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 1: EXECUTIVE SUMMARY
    # ============================================
    
    doc.add_heading('1. Executive Summary', level=1)
    
    # Calculate cumulative impact of top districts
    cumulative_missing = district_df_by_impact.head(TOP_N_DISTRICTS)['Missing_Weight'].sum()
    cumulative_pct = (cumulative_missing / total_missing * 100) if total_missing > 0 else 0
    top_5_cumulative_pct = district_df_by_impact.head(5)['Pct_of_Total_Missing'].sum()
    
    summary_text = f"""
    This report presents a comprehensive analysis of missing birth weight data in Sri Lanka, 
    stratified by district and ethnicity. The analysis follows IUPAC (International Union of 
    Pure and Applied Chemistry) standards for ethnic nomenclature to ensure international 
    consistency and scientific rigor.
    
    Key Findings:
    • Total Records Analyzed: {total_records:,}
    • Missing Birth Weight Records: {total_missing:,} ({total_missing/total_records*100:.2f}%)
    • Complete Records: {total_complete:,} ({total_complete/total_records*100:.2f}%)
    • Number of Districts Analyzed: {len(districts)}
    • Number of Ethnic Groups Identified: {len(unique_ethnicities)}
    • Focus Districts for Detailed Analysis: Top {TOP_N_DISTRICTS} districts by contribution to total missing records
    
    CRITICAL INSIGHT: The top {TOP_N_DISTRICTS} districts contribute {cumulative_pct:.1f}% of all missing 
    birth weight records nationally ({cumulative_missing:,} out of {total_missing:,} total missing records).
    The top 5 districts alone account for {top_5_cumulative_pct:.1f}% of all missing records.
    This concentration of missing data suggests that targeted interventions in these districts could 
    dramatically improve overall data completeness.
    
    Note: Birth weight is a critical indicator for newborn health, neonatal mortality, and 
    long-term developmental outcomes. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Public health intervention planning
    """
    
    doc.add_paragraph(summary_text)
    
    # ============================================
    # SECTION 2: IUPAC ETHNICITY CLASSIFICATION
    # ============================================
    
    doc.add_heading('2. IUPAC Ethnicity Classification System', level=1)
    doc.add_paragraph('The following ethnicity codes follow IUPAC standards for population genetics and vital statistics reporting:')
    
    # Create ethnicity table
    table = doc.add_table(rows=1, cols=3)
    table.style = 'Light Grid Accent 1'
    hdr_cells = table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'IUPAC Standard Name'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
        row_cells[2].text = config['iupac_name']
    
    doc.add_paragraph()
    
    # ============================================
    # SECTION 3: DISTRICT-WISE ANALYSIS - ALL DISTRICTS
    # ============================================
    
    doc.add_heading('3. District-Wise Missing Birth Weight Analysis', level=1)
    
    # ============================================
    # 3.1 ALL DISTRICTS SUMMARY TABLE
    # ============================================
    
    doc.add_heading('3.1 Complete District Summary (All 25 Districts)', level=2)
    doc.add_paragraph('The following table presents missing birth weight statistics for all districts, sorted by impact (percentage of total national missing records).')
    
    # Create table for ALL districts sorted by impact
    all_districts_table = doc.add_table(rows=1, cols=6)
    all_districts_table.style = 'Light Grid Accent 1'
    hdr_cells = all_districts_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Impact'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Weight Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    hdr_cells[5].text = '% of Total Missing Records'
    
    # Add all districts sorted by impact
    for idx, (_, row) in enumerate(district_df_by_impact.iterrows(), 1):
        row_cells = all_districts_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Total_Births']:,}"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[5].text = f"{row['Pct_of_Total_Missing']:.2f}%"
    
    doc.add_paragraph()
    
    # ============================================
    # 3.2 DISTRICTS WITH HIGHEST MISSING RATES (FOR CONTEXT)
    # ============================================
    
    doc.add_heading('3.2 Districts with Highest Missing Rates (Reference)', level=2)
    doc.add_paragraph('The following districts have the highest percentages of missing data, but may contribute less to the total missing count due to smaller population sizes.')
    
    rate_table = doc.add_table(rows=1, cols=4)
    rate_table.style = 'Light Grid Accent 1'
    hdr_cells = rate_table.rows[0].cells
    hdr_cells[0].text = 'Rank by Rate'
    hdr_cells[1].text = 'District'
    hdr_cells[2].text = 'Missing Rate (%)'
    hdr_cells[3].text = 'Missing Count'
    
    for idx, (_, row) in enumerate(district_df_by_rate.head(10).iterrows(), 1):
        row_cells = rate_table.add_row().cells
        row_cells[0].text = str(idx)
        row_cells[1].text = row['District']
        row_cells[2].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
    
    doc.add_paragraph()
    
    # ============================================
    # 3.3 CUMULATIVE IMPACT ANALYSIS
    # ============================================
    
    doc.add_heading('3.3 Cumulative Impact Analysis', level=2)
    
    cumulative_text = f"""
    The concentration of missing birth weight data across districts is highly uneven:
    
    • Top 5 Districts: {top_5_cumulative_pct:.1f}% of all missing records
    • Top 10 Districts: {district_df_by_impact.head(10)['Pct_of_Total_Missing'].sum():.1f}% of all missing records
    • Top {TOP_N_DISTRICTS} Districts: {cumulative_pct:.1f}% of all missing records
    
    This concentration indicates that a targeted intervention strategy focusing on high-impact districts
    could yield the greatest improvement in overall data completeness with efficient resource allocation.
    """
    
    doc.add_paragraph(cumulative_text)
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 4: DETAILED DISTRICT-ETHNICITY ANALYSIS (TOP IMPACT DISTRICTS)
    # ============================================
    
    doc.add_heading(f'4. Detailed District-Ethnicity Analysis (Top {TOP_N_DISTRICTS} Districts by Impact)', level=1)
    doc.add_paragraph(f'The following tables show the ethnicity distribution and missing birth weight patterns for the {TOP_N_DISTRICTS} districts that contribute most to national missing records, following IUPAC nomenclature. These districts represent the highest priority for data quality improvement interventions.')
    
    for district in top_districts_list[:TOP_N_DISTRICTS]:
        district_ethnic = ethnicity_district_df[ethnicity_district_df['District'] == district]
        
        if len(district_ethnic) > 0:
            district_total = district_df_by_impact[district_df_by_impact['District'] == district]['Total_Births'].values[0]
            district_missing = district_df_by_impact[district_df_by_impact['District'] == district]['Missing_Weight'].values[0]
            district_pct_missing = district_df_by_impact[district_df_by_impact['District'] == district]['Pct_of_Total_Missing'].values[0]
            district_rate = district_df_by_impact[district_df_by_impact['District'] == district]['Missing_Rate'].values[0]
            
            # Add district header with impact rank
            rank = district_df_by_impact[district_df_by_impact['District'] == district].index[0] + 1
            
            doc.add_heading(f'{district} (Impact Rank: #{rank})', level=2)
            doc.add_paragraph(f'Total Births: {district_total:,} | Missing Weight: {district_missing:,} ({district_rate:.2f}% of district) | {district_pct_missing:.1f}% of National Missing Records')
            
            # Create ethnicity table for district
            ethnic_table = doc.add_table(rows=1, cols=9)
            ethnic_table.style = 'Light Grid Accent 1'
            hdr_cells = ethnic_table.rows[0].cells
            hdr_cells[0].text = 'IUPAC Code'
            hdr_cells[1].text = 'Ethnicity'
            hdr_cells[2].text = 'Count'
            hdr_cells[3].text = '% of District'
            hdr_cells[4].text = 'Missing Weight'
            hdr_cells[5].text = 'Missing Rate (%)'
            hdr_cells[6].text = '% of District Missing Records'
            hdr_cells[7].text = '% of National Missing'
            hdr_cells[8].text = 'Rank in District'
            
            district_ethnic_sorted = district_ethnic.sort_values('Pct_of_District_Total', ascending=False)
            
            # Add rank by missing count within district
            district_ethnic_sorted = district_ethnic_sorted.copy()
            district_ethnic_sorted['Rank_In_District'] = district_ethnic_sorted['Missing_Weight'].rank(ascending=False, method='dense').astype(int)
            
            for _, row in district_ethnic_sorted.iterrows():
                row_cells = ethnic_table.add_row().cells
                row_cells[0].text = row['IUPAC_Code']
                row_cells[1].text = row['Ethnicity']
                row_cells[2].text = f"{row['Total_Mothers']:,}"
                row_cells[3].text = f"{row['Pct_of_District_Total']:.2f}%"
                row_cells[4].text = f"{row['Missing_Weight']:,}"
                row_cells[5].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[6].text = f"{row['Pct_of_District_Missing_Records']:.2f}%"
                row_cells[7].text = f"{row['Pct_of_National_Missing']:.2f}%"
                row_cells[8].text = str(row['Rank_In_District'])
            
            # Add district summary
            doc.add_paragraph()
            most_prevalent = district_ethnic_sorted.iloc[0]
            highest_missing = district_ethnic_sorted.loc[district_ethnic_sorted['Missing_Rate_in_Ethnic'].idxmax()]
            largest_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_District_Missing_Records'].idxmax()]
            largest_national_contributor = district_ethnic_sorted.loc[district_ethnic_sorted['Pct_of_National_Missing'].idxmax()]
            
            summary_para = doc.add_paragraph()
            summary_para.add_run('District Summary:').bold = True
            doc.add_paragraph(f'• Most prevalent ethnicity: {most_prevalent["Ethnicity"]} ({most_prevalent["IUPAC_Code"]}) - {most_prevalent["Pct_of_District_Total"]:.1f}% of district')
            doc.add_paragraph(f'• Highest missing rate: {highest_missing["Ethnicity"]} ({highest_missing["IUPAC_Code"]}) - {highest_missing["Missing_Rate_in_Ethnic"]:.1f}% missing birth weight')
            doc.add_paragraph(f'• Largest contributor to district missing data: {largest_contributor["Ethnicity"]} ({largest_contributor["IUPAC_Code"]}) - {largest_contributor["Pct_of_District_Missing_Records"]:.1f}% of district missing records')
            doc.add_paragraph(f'• Largest contributor to national missing data: {largest_national_contributor["Ethnicity"]} ({largest_national_contributor["IUPAC_Code"]}) - {largest_national_contributor["Pct_of_National_Missing"]:.1f}% of national missing records')
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 5: ETHNICITY-SPECIFIC ANALYSIS
    # ============================================
    
    doc.add_heading('5. Ethnicity-Specific Analysis', level=1)
    
    # Calculate overall ethnicity statistics
    ethnicity_overall = ethnicity_district_df.groupby(['Ethnicity', 'IUPAC_Code', 'IUPAC_Name']).agg({
        'Total_Mothers': 'sum',
        'Missing_Weight': 'sum',
        'Pct_of_National_Missing': 'sum'
    }).reset_index()
    ethnicity_overall['Missing_Rate'] = (ethnicity_overall['Missing_Weight'] / ethnicity_overall['Total_Mothers'] * 100)
    ethnicity_overall = ethnicity_overall.sort_values('Missing_Rate', ascending=False)
    
    doc.add_heading('5.1 Overall Ethnicity Statistics (Top Impact Districts Only)', level=2)
    doc.add_paragraph(f'Note: These statistics represent only the top {TOP_N_DISTRICTS} districts by impact.')
    
    overall_table = doc.add_table(rows=1, cols=6)
    overall_table.style = 'Light Grid Accent 1'
    hdr_cells = overall_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnicity'
    hdr_cells[2].text = 'Total Births'
    hdr_cells[3].text = 'Missing Weight Records'
    hdr_cells[4].text = 'Missing Rate (%)'
    hdr_cells[5].text = '% of National Missing'
    
    for _, row in ethnicity_overall.iterrows():
        row_cells = overall_table.add_row().cells
        row_cells[0].text = row['IUPAC_Code']
        row_cells[1].text = row['Ethnicity']
        row_cells[2].text = f"{row['Total_Mothers']:,}"
        row_cells[3].text = f"{row['Missing_Weight']:,}"
        row_cells[4].text = f"{row['Missing_Rate']:.2f}%"
        row_cells[5].text = f"{row['Pct_of_National_Missing']:.2f}%"
    
    # Add ethnicity-specific district analysis
    doc.add_heading('5.2 Ethnicity-Specific District Analysis (Top Impact Districts)', level=2)
    
    for ethnicity in list(ETHNICITIES.keys())[:6]:  # Show first 6 ethnicities
        ethnic_data = ethnicity_district_df[ethnicity_district_df['Ethnicity'] == ethnicity]
        
        if len(ethnic_data) > 0:
            ethnic_total = ethnic_data['Total_Mothers'].sum()
            ethnic_missing = ethnic_data['Missing_Weight'].sum()
            ethnic_rate = (ethnic_missing / ethnic_total * 100) if ethnic_total > 0 else 0
            ethnic_national_pct = ethnic_data['Pct_of_National_Missing'].sum()
            
            doc.add_heading(f'{ETHNICITIES[ethnicity]["full_name"]} ({ETHNICITIES[ethnicity]["code"]})', level=3)
            doc.add_paragraph(f'Overall Statistics: Total Births: {ethnic_total:,} | Missing Weight: {ethnic_missing:,} ({ethnic_rate:.2f}%) | {ethnic_national_pct:.1f}% of National Missing Records')
            
            # Create table for districts with highest missing counts for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Counts for this Ethnicity:', style='List Bullet')
            count_table = doc.add_table(rows=1, cols=5)
            count_table.style = 'Light Grid Accent 1'
            hdr_cells = count_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_count = ethnic_data.sort_values('Missing_Weight', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_count.iterrows():
                row_cells = count_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
            
            # Table for districts with highest missing rates for this ethnicity
            doc.add_paragraph('Districts with Highest Missing Rates for this Ethnicity:', style='List Bullet')
            rate_table = doc.add_table(rows=1, cols=5)
            rate_table.style = 'Light Grid Accent 1'
            hdr_cells = rate_table.rows[0].cells
            hdr_cells[0].text = 'District'
            hdr_cells[1].text = 'Total Births'
            hdr_cells[2].text = 'Missing Weight'
            hdr_cells[3].text = 'Missing Rate (%)'
            hdr_cells[4].text = '% of Ethnic Missing'
            
            ethnic_sorted_by_rate = ethnic_data.sort_values('Missing_Rate_in_Ethnic', ascending=False).head(10)
            
            for _, row in ethnic_sorted_by_rate.iterrows():
                row_cells = rate_table.add_row().cells
                row_cells[0].text = row['District']
                row_cells[1].text = f"{row['Total_Mothers']:,}"
                row_cells[2].text = f"{row['Missing_Weight']:,}"
                row_cells[3].text = f"{row['Missing_Rate_in_Ethnic']:.2f}%"
                row_cells[4].text = f"{(row['Missing_Weight']/ethnic_missing*100):.2f}%"
            
            doc.add_paragraph()
    
    doc.add_page_break()
    
    # ============================================
    # SECTION 6: CONCLUSIONS AND RECOMMENDATIONS
    # ============================================
    
    doc.add_heading('6. Conclusions and Recommendations', level=1)
    
    # Find key insights
    top_impact_district = district_df_by_impact.iloc[0]
    top_5_impact = district_df_by_impact.head(5)
    top_5_cumulative_pct = top_5_impact['Pct_of_Total_Missing'].sum()
    
    worst_ethnicity_by_rate = ethnicity_overall.iloc[0] if len(ethnicity_overall) > 0 else None
    top_ethnic_contributor = ethnicity_overall.nlargest(1, 'Pct_of_National_Missing').iloc[0] if len(ethnicity_overall) > 0 else None
    
    conclusions = f"""
    6.1 Key Findings
    
    • Data Completeness: {total_complete/total_records*100:.2f}% of birth records have complete birth weight data,
      indicating {total_missing/total_records*100:.2f}% of records require data quality improvement.
    
    • Concentration of Missing Data: The top {TOP_N_DISTRICTS} districts contribute {cumulative_pct:.1f}% of all missing 
      birth weight records nationally. The top 5 districts alone contribute {top_5_cumulative_pct:.1f}% of all missing records.
      This high concentration suggests that targeted interventions could dramatically improve overall data completeness.
    
    • Highest Impact District: {top_impact_district['District']} contributes the largest share of missing records 
      ({top_impact_district['Pct_of_Total_Missing']:.1f}% of total missing, {top_impact_district['Missing_Weight']:,} records).
      This district should be the highest priority for intervention.
    """
    
    if worst_ethnicity_by_rate is not None:
        conclusions += f"""
    • Ethnic Disparities by Rate (in top impact districts): {worst_ethnicity_by_rate['Ethnicity']} ({worst_ethnicity_by_rate['IUPAC_Code']}) has the highest 
      missing rate for birth weight at {worst_ethnicity_by_rate['Missing_Rate']:.2f}%, indicating potential systematic bias 
      in data collection across ethnic groups within high-impact districts.
        """
    
    if top_ethnic_contributor is not None:
        conclusions += f"""
    • Ethnic Disparities by Contribution (in top impact districts): {top_ethnic_contributor['Ethnicity']} ({top_ethnic_contributor['IUPAC_Code']}) 
      contributes the largest share ({top_ethnic_contributor['Pct_of_National_Missing']:.1f}%) of all missing birth weight records nationally.
        """
    
    conclusions += """
    • Public Health Implications: Missing birth weight data affects the accuracy of low birth weight surveillance, 
      neonatal mortality assessments, and maternal and child health program evaluations. Birth weight is a critical 
      indicator for:
      - Low birth weight (<2500g) prevalence monitoring
      - Neonatal mortality risk assessment
      - Maternal nutrition intervention effectiveness
      - Newborn health outcomes tracking
    
    6.2 Recommendations (Prioritized by Impact)
    
    1. Focus on High-Impact Districts: Prioritize data quality improvement efforts on the top 5 districts that
       contribute the largest share of missing records. These districts alone account for {top_5_cumulative_pct:.1f}% 
       of all missing birth weight data nationally.
    """
    
    conclusions += f"""
    2. District-Specific Interventions: Implement specialized data collection protocols in the highest impact districts:
    """
    
    # Add top 5 impact districts
    for idx, (_, row) in enumerate(top_5_impact.iterrows(), 1):
        conclusions += f"       • {row['District']}: {row['Pct_of_Total_Missing']:.1f}% of total missing\n"
    
    if worst_ethnicity_by_rate is not None and top_ethnic_contributor is not None:
        conclusions += f"""
    3. Ethnicity-Specific Interventions: Develop targeted data quality improvement programs for ethnic groups 
       with both high missing rates and high contribution to national missing data, focusing on:
       • {worst_ethnicity_by_rate['Ethnicity']}: {worst_ethnicity_by_rate['Missing_Rate']:.1f}% missing rate
       • {top_ethnic_contributor['Ethnicity']}: {top_ethnic_contributor['Pct_of_National_Missing']:.1f}% of national missing
        """
    
    conclusions += """
    4. IUPAC Compliance: Maintain adherence to IUPAC ethnic nomenclature standards to ensure international 
       comparability and scientific rigor.
    
    5. Regular Monitoring: Establish quarterly data quality monitoring systems with a focus on the high-impact
       districts identified in this report.
    
    6. Capacity Building: Provide training for healthcare workers on the importance of accurate birth weight 
       measurement and documentation, with intensified efforts in high-impact districts.
    
    7. Equipment Maintenance: Ensure regular calibration and maintenance of weighing scales in healthcare 
       facilities, prioritizing high-volume facilities in high-impact districts.
    
    8. Electronic Health Records: Implement or strengthen electronic health record systems with mandatory 
       birth weight fields, with phased implementation starting in high-impact districts.
    """
    
    doc.add_paragraph(conclusions)
    
    # Add methodology section
    doc.add_heading('7. Methodology', level=1)
    
    methodology = f"""
    This analysis was conducted using vital statistics data from Sri Lanka. The methodology follows 
    IUPAC standards for ethnic classification and WHO guidelines for demographic data collection.
    
    Data Sources:
    • Birth Registration Records: {total_records:,} records
    • Time Period: Full dataset analysis
    • Geographic Coverage: {len(districts)} districts
    • Focus Districts for Detailed Analysis: Top {TOP_N_DISTRICTS} districts by contribution to total missing records
    
    Key Metrics Defined:
    
    1. Missing Rate: Percentage of records missing birth weight within a specific group
       Formula: (Missing Weight / Total Births) × 100
    
    2. Percentage of Total Missing Records: What proportion of ALL national missing records 
       comes from a specific district or ethnicity
       Formula: (Missing Weight in Group / Total National Missing) × 100
    
    IMPORTANT: 
    • Section 3.1 presents statistics for ALL {len(districts)} districts
    • Districts were selected for detailed ethnicity analysis (Section 4) based on their 
      contribution to total missing records (Percentage of Total Missing Records), NOT by 
      Missing Rate. This ensures that the analysis focuses on districts that have the 
      greatest potential impact on overall data completeness.
    
    3. Percentage of District Missing Records: What proportion of a district's missing records 
       comes from a specific ethnicity
       Formula: (Missing Weight for Ethnicity in District / Total District Missing) × 100
    
    4. Percentage of National Missing: What proportion of ALL national missing records comes 
       from a specific ethnicity-district combination
    
    Birth Weight Definition:
    Birth weight is the first weight of the newborn taken within the first hours of life, 
    measured in grams. Complete birth weight data is essential for:
    • Low birth weight surveillance (<2500g)
    • Very low birth weight tracking (<1500g)
    • Extremely low birth weight monitoring (<1000g)
    • Neonatal mortality risk assessment
    • Maternal nutrition program evaluation
    • Demographic transition analysis
    • Newborn health outcomes research
    
    Birth Weight Categories (WHO):
    • Low Birth Weight: <2500g
    • Normal Birth Weight: 2500g - 3999g
    • High Birth Weight: ≥4000g
    
    Ethnic Classification:
    Ethnicities were classified according to IUPAC standards using the following codes:
    """
    
    doc.add_paragraph(methodology)
    
    # Add IUPAC code reference
    ref_table = doc.add_table(rows=1, cols=2)
    ref_table.style = 'Light Grid Accent 1'
    hdr_cells = ref_table.rows[0].cells
    hdr_cells[0].text = 'IUPAC Code'
    hdr_cells[1].text = 'Ethnic Group'
    
    for ethnicity, config in ETHNICITIES.items():
        row_cells = ref_table.add_row().cells
        row_cells[0].text = config['code']
        row_cells[1].text = config['full_name']
    
    # Save the document
    filename = f'Missing_Birth_Weight_Analysis_All_Districts_{datetime.now().strftime("%Y%m%d_%H%M%S")}.docx'
    doc.save(filename)
    print(f"\n✅ Word document saved as: {filename}")
    
    # ============================================
    # STEP 5: DISPLAY SUMMARY IN CONSOLE
    # ============================================
    
    print("\n" + "=" * 100)
    print(f"📊 FINAL SUMMARY: Missing Birth Weight Analysis by District and Ethnicity")
    print("=" * 100)
    
    print(f"\n✅ Analysis Complete!")
    print(f"✅ Word Document Generated: {filename}")
    print(f"\nOverall Statistics:")
    print(f"   • Total Districts: {len(districts)}")
    print(f"   • Focus Districts for Detailed Analysis: {TOP_N_DISTRICTS} (by contribution to total missing)")
    print(f"   • Total Births: {total_records:,}")
    print(f"   • Total Missing Birth Weight: {total_missing:,} ({total_missing/total_records*100:.2f}%)")
    
    print(f"\n🏆 Top {TOP_N_DISTRICTS} Districts by IMPACT (% of Total Missing Records):")
    for idx, (_, row) in enumerate(district_df_by_impact.head(TOP_N_DISTRICTS).iterrows(), 1):
        print(f"   {idx}. {row['District']}: {row['Pct_of_Total_Missing']:.2f}% of total missing ({row['Missing_Weight']:,} records) - Missing Rate: {row['Missing_Rate']:.2f}%")
    
    print(f"\n📊 Cumulative Impact:")
    print(f"   • Top 5 districts account for {district_df_by_impact.head(5)['Pct_of_Total_Missing'].sum():.1f}% of all missing records")
    print(f"   • Top 10 districts account for {district_df_by_impact.head(10)['Pct_of_Total_Missing'].sum():.1f}% of all missing records")
    print(f"   • Top {TOP_N_DISTRICTS} districts account for {cumulative_pct:.1f}% of all missing records")
    
    print(f"\n🏆 Top 5 Districts with Highest Missing RATE (for context):")
    for _, row in district_df_by_rate.head(5).iterrows():
        print(f"   • {row['District']}: {row['Missing_Rate']:.2f}% missing rate ({row['Missing_Weight']:,} records)")
    
    if len(ethnicity_overall) > 0:
        print(f"\n🏆 Top 5 Ethnicities with Highest Missing RATE (in top impact districts):")
        for _, row in ethnicity_overall.head(5).iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Missing_Rate']:.2f}% ({row['Missing_Weight']:,}/{row['Total_Mothers']:,})")
        
        print(f"\n🏆 Top 5 Ethnicities by % of Total National Missing (in top impact districts):")
        ethnicity_by_pct = ethnicity_overall.nlargest(5, 'Pct_of_National_Missing')
        for _, row in ethnicity_by_pct.iterrows():
            print(f"   • {row['Ethnicity']} ({row['IUPAC_Code']}): {row['Pct_of_National_Missing']:.2f}% of national missing ({row['Missing_Weight']:,} records)")
    
    print(f"\n💡 KEY INSIGHT: Focus interventions on the top {TOP_N_DISTRICTS} districts to address {cumulative_pct:.1f}% of all missing birth weight records.")
    print(f"   Complete district-level data for all {len(districts)} districts is available in Section 3.1 of the report.")

else:
    print("\n⚠️ No ethnicity data found. Please check the ethnicity column values.")
    print(f"Sample values from {MOTHER_RACE_COL}:")
    print(df[MOTHER_RACE_COL].value_counts().head(10))

print("\n✅ Analysis Complete!")

🏥 MISSING BIRTH WEIGHT ANALYSIS BY DISTRICT AND ETHNICITY

📊 STEP 1: Missing Birth Weight Analysis
--------------------------------------------------------------------------------

Overall Statistics:
   • Total records: 363,881
   • Missing birth weight: 45,303 (12.45%)
   • Complete birth weight: 318,578 (87.55%)

📊 STEP 2: Standardizing Ethnicities (IUPAC Standards)
--------------------------------------------------------------------------------

Unique ethnicities found in dataset: ['Bharatha', 'Burgher', 'Indian Moor', 'Indian Tamil', 'Malay', 'Other Foreigners', 'Other Srilankans', 'Pakistan Moor', 'Sinhalese', 'Srilankan Chetty', 'Srilankan Moor', 'Srilankan Tamil']

📊 STEP 3: District-Level Missing Birth Weight Analysis
--------------------------------------------------------------------------------

All Districts Summary (Total: 25 districts):
   • Total records across all districts: 363,881
   • Total missing records: 45,303

Top 15 districts by IMPACT (% of Total Missing Rec